# Predicción de horas hombre — Entradas a almacén (handhelds)

**Objetivo de negocio:** estimar las **horas hombre** de trabajo de almacén que se necesitarán, a
partir del histórico de entradas (`ds_entradas`), para planear personal y turnos.

Este notebook **reutiliza** la carga de datos de `fuente_unica_entradas.ipynb` (§6, BD Zeus con hora
real) y **replica la estructura y las gráficas** de `Timeseries.ipynb` (SARIMA + Prophet, en *dark
mode*), adaptadas de una serie **mensual** a una serie **horaria**.

> **Variable objetivo — "horas hombre de presencia":** número de usuarios distintos activos en cada
> hora reloj. Una persona presente y trabajando en una hora ⇒ 1 hora-hombre. Es un **proxy de
> presencia** (no horas de nómina), pero es exactamente lo que se programa contra turnos.

# Bibliotecas y funciones

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")  # evita sobre-suscripción de BLAS en la rejilla SARIMAX
import itertools

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

import statsmodels
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet

from sklearn.metrics import mean_squared_error, mean_absolute_error
from joblib import Parallel, delayed

from sqlalchemy import create_engine, text

In [ ]:
#@title Dark Mode
plt.style.use("dark_background")
plt.rcParams.update({
    "figure.facecolor": "#111111", "axes.facecolor": "#111111", "savefig.facecolor": "#111111",
    "axes.edgecolor": "white", "axes.labelcolor": "white", "text.color": "white",
    "xtick.color": "white", "ytick.color": "white",
    "grid.color": "lightgray", "grid.linestyle": "--", "grid.linewidth": 0.5,
    "lines.linewidth": 1.5, "lines.markersize": 5, "font.size": 10,
    "figure.figsize": (8, 6), "savefig.dpi": 150, "savefig.bbox": "tight",
    "legend.edgecolor": "white", "legend.frameon": True, "legend.fancybox": False,
})
sns.set_style("darkgrid", {
    "axes.facecolor": "#111111", "figure.facecolor": "#111111", "grid.color": "#333333",
    "text.color": "white", "axes.labelcolor": "white", "xtick.color": "white", "ytick.color": "white",
})

In [ ]:
def _oscurecer(fig):
    """Fuerza modo oscuro en figuras que crean su propia Figure (seasonal_decompose,
    plot_diagnostics, Prophet.plot / plot_components), que no heredan bien los rcParams."""
    fig.patch.set_facecolor("#111111")
    for ax in fig.get_axes():
        ax.set_facecolor("#111111")
        ax.tick_params(colors="white")
        for spine in ax.spines.values():
            spine.set_edgecolor("white")
        ax.xaxis.label.set_color("white"); ax.yaxis.label.set_color("white")
        ax.title.set_color("white")
        # Recolorea artistas negros (p. ej. los puntos de historia de Prophet) para que se vean.
        for ln in ax.get_lines():
            r, g, b, _ = mcolors.to_rgba(ln.get_color())
            if r < 0.2 and g < 0.2 and b < 0.2:
                ln.set_color("#cccccc")
        leg = ax.get_legend()
        if leg is not None:
            for t in leg.get_texts():
                t.set_color("white")
    fig.tight_layout()
    return fig

In [ ]:
def p_value_adfuller(data):
        fuller_test = adfuller(data)
        print("P-value: ", fuller_test[1])
        if fuller_test[1] <= 0.05:
            print("Se rechaza la hipótesis nula, los datos son estacionarios")
        else:
            print("No se rechaza la hipóteis nula, los datos no son estacionarios")

## Constantes de adaptación (mensual → horario)

Todo el mapeo de `Timeseries.ipynb` (mensual, `period=12`) a esta serie **horaria** se concentra aquí.

In [ ]:
PERIODO        = 24            # estacionalidad diaria (horas) -> ranura estacional de SARIMA
PERIODO_SEMANA = 168           # estacionalidad semanal (horas) -> vía exóg. de Fourier
FREQ           = "h"           # frecuencia horaria (minúscula; "H" está deprecada en pandas)
LAGS           = 48            # 2 días, para ver el peine diario en ACF/PACF
VENTANA_MM     = 24            # media móvil de 24 h
HORAS_LAB      = range(6, 21)  # ventana laboral observada (06–20h)
DIAS_TEST      = 7             # holdout = última semana completa
PERIODOS_FUT   = DIAS_TEST * 24    # 168 h: horizonte de pronóstico (igual SARIMA y Prophet)
K_FOURIER      = 3             # armónicos de la estacionalidad semanal

# Lectura y construcción de la serie de tiempo

## 1. Lectura de la fuente con hora — tabla `ds_entradas` (BD Zeus)

Reutilizamos el cargador de `fuente_unica_entradas.ipynb` §6. El **API SGA** entrega las fechas
**sin hora**, por lo que para un análisis horario leemos directamente la BD Zeus
(`ZEUS_DATABASE_URL`), donde `fecha_inicio` es `TIMESTAMP` con `HH:MM:SS` reales. Usamos
`fecha_inicio` como el instante en que el empleado arrancó la tarea en la handheld.

In [ ]:
# --- Lectura de la fuente CON hora: tabla `ds_entradas` de la BD Zeus (verbatim de §6) ---
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

ZEUS_URL = os.getenv("ZEUS_DATABASE_URL")
assert ZEUS_URL, "Falta ZEUS_DATABASE_URL (.env o variable de entorno) para leer la BD Zeus."

_cols = [
    "fecha_gen", "fecha_inicio", "fecha_fin",
    "usuario", "usuario_gen", "trabajador",
    "maquina", "maquina_gen", "status", "almacen",
]
_engine = create_engine(ZEUS_URL)
ent = pd.read_sql(
    text(f"SELECT {', '.join(_cols)} FROM ds_entradas"),
    _engine,
    parse_dates=["fecha_gen", "fecha_inicio", "fecha_fin"],
)
print(f"ds_entradas: {len(ent):,} filas | "
      f"fecha_inicio {ent['fecha_inicio'].min():%d-%b-%Y %H:%M} → {ent['fecha_inicio'].max():%d-%b-%Y %H:%M}")

# --- Variables temporales sobre fecha_inicio (= arranque de la tarea en la handheld) ---
ent = ent[ent["fecha_inicio"].notna()].copy()

# Usuarios de negocio: excluimos cuentas de sistema y nulos (configurable).
SYS_USERS = {"ADMIN", "", None}
ent = ent[ent["usuario"].notna() & ~ent["usuario"].isin(SYS_USERS)].copy()

ent["hora"] = ent["fecha_inicio"].dt.hour
ent["fecha"] = ent["fecha_inicio"].dt.normalize()  # día (sin hora)
DOW_ES = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
ent["dia_sem"] = pd.Categorical(
    ent["fecha_inicio"].dt.dayofweek.map(dict(enumerate(DOW_ES))),
    categories=DOW_ES, ordered=True,
)

# Duración de la tarea (min), acotada (cap 30 min) para no contaminar métricas de tiempo ocupado.
DUR_CAP_MIN = 30
ent["dur_min"] = (ent["fecha_fin"] - ent["fecha_inicio"]).dt.total_seconds() / 60.0
ent.loc[(ent["dur_min"] < 0) | (ent["dur_min"] > DUR_CAP_MIN), "dur_min"] = np.nan

print(f"Tareas: {len(ent):,} | usuarios: {ent['usuario'].nunique()} | handhelds: {ent['maquina'].nunique()}")

## Definición de la variable objetivo — horas hombre de presencia (por hora)

Para cada **hora reloj** contamos cuántos **usuarios distintos** estuvieron activos (= horas-hombre
acumuladas en esa hora). Se dejan comentadas dos alternativas intercambiables: **B** tiempo
realmente laborado (`dur_min/60`, diminuto y ruidoso por el cap de 30 min) y **C** volumen de tareas.

In [ ]:
# Sello de tiempo a la hora (floor) y conteo de usuarios distintos por hora reloj.
ent["ts"] = ent["fecha_inicio"].dt.floor(FREQ)
hh = ent.groupby("ts")["usuario"].nunique()
hh.name = "HorasHombre"

# --- Alternativas (descomentar para intercambiar la variable objetivo) ---
# B) Tiempo laborado (horas) = suma de duraciones / 60  (diminuto y ruidoso, cap 30 min)
# hh = ent.groupby("ts")["dur_min"].sum().div(60).rename("HorasHombre")
# C) Volumen de tareas por hora (proxy de carga)
# hh = ent.groupby("ts").size().rename("HorasHombre")

print(f"Horas con actividad: {len(hh):,} | "
      f"rango {hh.index.min():%d-%b-%Y %H:%M} → {hh.index.max():%d-%b-%Y %H:%M}")
hh.head()

## Asegurar el orden y la continuidad de la serie de tiempo

La serie debe ser **monótona** y **continua** en su frecuencia. La serie horaria tiene **huecos
reales** (horas sin ninguna entrada: noches, fines de semana, cierres) que rellenamos con **0**
(cero presencia es un valor verdadero, no un dato faltante; usar `ffill` inventaría trabajo fantasma).

In [ ]:
hh.index.is_monotonic_increasing

In [ ]:
# Horas esperadas (rejilla horaria continua) vs. horas presentes -> huecos a rellenar con 0.
obsEsperadas = pd.date_range(start=hh.index.min().floor("D"), end=hh.index.max(), freq=FREQ)
faltantes = obsEsperadas[~obsEsperadas.isin(hh.index)]
print(f"Horas esperadas: {len(obsEsperadas):,} | con actividad: {len(hh):,} | "
      f"huecos (0 h-hombre): {len(faltantes):,}")
faltantes[:10]

In [ ]:
serie = hh.reindex(obsEsperadas, fill_value=0).astype(float)
datos = serie.to_frame("HorasHombre")
datos.index.freq = FREQ   # frecuencia asignada DESPUÉS del reindex continuo
print(f"Serie continua: {len(datos):,} horas | freq={datos.index.freq} | "
      f"{(datos['HorasHombre'] == 0).mean():.0%} de ceros (noche/fin de semana/cierres)")
datos.head()

In [ ]:
end_ts = datos.index.max()
split_ts = end_ts - pd.Timedelta(hours=PERIODOS_FUT)   # holdout = última semana (168 h)
print("Fin de la serie :", end_ts)
print("Corte train/test:", split_ts)

# División de la serie en entrenamiento y prueba

Reservamos la **última semana completa** (168 h) como prueba — análogo horario de reservar un ciclo
estacional en `Timeseries.ipynb`. Mismo corte y horizonte para SARIMA y Prophet.

In [ ]:
train = datos.loc[:split_ts, "HorasHombre"].copy()
test  = datos.loc[split_ts + pd.Timedelta(hours=1):, "HorasHombre"].copy()
train.index.freq = FREQ
test.index.freq = FREQ
assert len(test) == PERIODOS_FUT, f"test debe tener {PERIODOS_FUT} h, tiene {len(test)}"
print(f"train: {len(train):,} h ({train.index.min():%d-%b %H:%M} → {train.index.max():%d-%b %H:%M})")
print(f"test : {len(test):,} h ({test.index.min():%d-%b %H:%M} → {test.index.max():%d-%b %H:%M})")

# Exploración y Visualización de la Serie Temporal

In [ ]:
ax = train.plot(figsize=(12, 4), label="entrenamiento",
                title="Horas hombre por hora — Entradas a almacén (handhelds)")
ax.set_xlabel("fecha"); ax.set_ylabel("horas-hombre (usuarios activos)")
plt.legend(); plt.show()

**¿Es la serie estacionaria?** La media móvil de 24 h ayuda a ver la tendencia bajo el fuerte
ciclo diario.

In [ ]:
ax = train.plot(figsize=(12, 4), label="Horas hombre/h", alpha=0.5)
train.rolling(VENTANA_MM).mean().plot(ax=ax, label=f"Media móvil {VENTANA_MM} h")
plt.title(f"Horas hombre por hora con Media Móvil {VENTANA_MM} h")
plt.legend(); plt.show()

`seasonal_decompose` separa la serie en **tendencia**, **temporalidad** (perfil diario de 24 h)
y **residuos**. Usamos `period=24` (estacionalidad diaria) y modelo **aditivo** (los ceros nocturnos
hacen inapropiado el multiplicativo).

In [ ]:
decomposition = seasonal_decompose(train, model="additive", period=PERIODO, extrapolate_trend="freq")
fig = decomposition.plot(); fig.set_size_inches(11, 6)
_oscurecer(fig); plt.show()

In [ ]:
residuals = decomposition.resid
residuals.plot(figsize=(11, 2.5))
plt.ylabel("residuos"); plt.xlabel("fecha"); plt.show()

In [ ]:
_ = residuals.dropna().plot.hist(figsize=(5, 4), bins=30)
plt.title("Histograma de residuos"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
plot_acf(residuals.dropna(),  lags=LAGS, ax=axes[0])
plot_pacf(residuals.dropna(), lags=LAGS, ax=axes[1])
plt.show()

Las correlaciones significativas (sobre todo el **peine en 24 y 48 h**) indican que modelar la
serie con un **SARIMA estacional (s=24)** es buena idea. La estacionalidad **semanal (168 h)** se
añade como **exógenas de Fourier**: una sola ranura estacional no alcanza para 24 y 168 a la vez.

# Construcción del modelo SARIMA

## Estacionariedad de la serie (parámetro d)

**Test Dickey-Fuller** — H0: la serie *no* es estacionaria (si *p* es pequeña, se rechaza H0).

In [ ]:
p_value_adfuller(train)

La serie horaria ya es **estacionaria** (el ciclo diurno la hace revertir a la media), así que
**d = 0**. Mostramos igualmente la 1ª diferencia de forma ilustrativa.

In [ ]:
train_1diff = train - train.shift(1)
train_1diff.plot(figsize=(11, 2.5), label="Primera diferenciación (ilustrativa)")
plt.title("Primera diferenciación de la serie"); plt.legend(); plt.show()

In [ ]:
p_value_adfuller(train_1diff.dropna())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
plot_acf(train_1diff.dropna(),  lags=LAGS, ax=axes[0])
plot_pacf(train_1diff.dropna(), lags=LAGS, ax=axes[1])
plt.show()

In [ ]:
train_24diff = train - train.shift(PERIODO)
train_24diff.plot(figsize=(11, 2.5), label="Diferenciación estacional (24 h)")
plt.title("Primera diferenciación de temporalidad diaria (lag 24)"); plt.legend(); plt.show()

In [ ]:
p_value_adfuller(train_24diff.dropna())

Por tanto **d = 0** y **D = 1** (una diferencia estacional en el lag 24). `D = 2` nunca.

## Estacionalidad semanal vía exógenas de Fourier (período 168)

SARIMA solo tiene **una** ranura estacional (la usamos para 24 h). El ciclo **semanal** (caída de fin
de semana) se inyecta como **términos de Fourier** (seno/coseno de período 168). La base es
determinista en el tiempo: se construye sobre un índice **continuo** que cubre train + futuro y luego
se **rebana** (no reiniciar `t` para el futuro, o la fase del fin de semana saldría mal).

In [ ]:
def fourier_terms(index, period, K):
    """Matriz de términos de Fourier (seno/coseno) para una estacionalidad de longitud `period`."""
    t = np.arange(len(index))
    out = {}
    for k in range(1, K + 1):
        out[f"s{period}_sin{k}"] = np.sin(2 * np.pi * k * t / period)
        out[f"s{period}_cos{k}"] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(out, index=index)

exog = fourier_terms(datos.index, PERIODO_SEMANA, K_FOURIER)   # sobre TODA la serie (continua)
exog_train = exog.loc[:split_ts]
exog_fut   = exog.loc[split_ts + pd.Timedelta(hours=1):]
print(f"exóg. Fourier semanal: {exog.shape[1]} columnas | "
      f"train {exog_train.shape[0]} | futuro {exog_fut.shape[0]}")
exog.head()

## Orden del modelo SARIMA (p, d, q) × (P, D, Q, 24)

Buscamos el orden con una rejilla **acotada** (p, q, P, Q ∈ {0, 1}; d = 0, D = 1, s = 24 ⇒ 16
ajustes), rankeada por AIC. Con ~4 000 puntos y s = 24 una rejilla mayor sería de minutos a horas;
mantenemos el espacio pequeño y cada ajuste protegido con `try/except`.

In [ ]:
p = q = range(0, 2)
pdq = list(itertools.product(p, [0], q))                              # d = 0 fijo
P = Q = range(0, 2)
PDQs = [(x[0], 1, x[1], PERIODO) for x in itertools.product(P, Q)]    # D = 1, s = 24
print("No estacional (p,d,q):", pdq)
print("Estacional (P,D,Q,s) :", PDQs)

In [ ]:
%%time
def fit_sarimax_model(parametros, parametros_s):
    try:
        modelo = SARIMAX(train, order=parametros, seasonal_order=parametros_s,
                         enforce_stationarity=False, enforce_invertibility=False)
        res = modelo.fit(disp=False, maxiter=50)
        return (parametros, parametros_s), res.aic
    except Exception:
        return (parametros, parametros_s), np.inf

resultados_grid = Parallel(n_jobs=4)(
    delayed(fit_sarimax_model)(o, s) for o in pdq for s in PDQs
)
aic_modelos = dict(resultados_grid)

In [ ]:
mejor = min(aic_modelos, key=aic_modelos.get)
print("Mejor orden por AIC (rejilla sin exóg.):", mejor, "| AIC =", round(aic_modelos[mejor], 1))
aic_modelos

La rejilla orienta el orden, pero el **forecaster final** usa un orden **parsimonioso y
robusto** —`(1,0,1)(0,1,1,24)`, estilo *airline*— **con** las exógenas de Fourier semanal.
Hard-codear el orden rico del tutorial original `(1,1,2)(2,1,2,·)` sobre solo ~21 ciclos semanales
sería sobre-parametrizar.

In [ ]:
modelo = SARIMAX(train, exog=exog_train,
                 order=(1, 0, 1), seasonal_order=(0, 1, 1, PERIODO),
                 enforce_stationarity=False, enforce_invertibility=False)
resultados = modelo.fit(disp=False, maxiter=50)
print(resultados.summary().tables[1])

# Validación

## Análisis de los residuos del modelo

In [ ]:
fig = resultados.plot_diagnostics(figsize=(15, 8))
_oscurecer(fig); plt.show()

## Predicción y evaluación del modelo (168 h)

In [ ]:
pred = resultados.get_forecast(steps=len(test), exog=exog_fut)
pred_mean = pred.predicted_mean.clip(lower=0)   # las horas-hombre no pueden ser negativas
pred_mean.index = test.index                    # alinear al rango de prueba
pred_mean.head()

In [ ]:
pred_IC = pred.conf_int().clip(lower=0)
pred_IC.index = test.index
pred_IC.tail()

In [ ]:
ax = datos["HorasHombre"].plot(label="Observaciones", figsize=(12, 4), alpha=0.6)
pred_mean.plot(ax=ax, label="Pronóstico SARIMA", color="#FF7043")
ax.fill_between(pred_IC.index, pred_IC.iloc[:, 0], pred_IC.iloc[:, 1], color="#FF7043", alpha=0.25)
plt.title("SARIMA — observaciones vs. pronóstico (rango completo)")
plt.legend(); plt.show()

In [ ]:
zoom = split_ts - pd.Timedelta(days=14)
ax = datos.loc[zoom:, "HorasHombre"].plot(label="Observaciones", figsize=(12, 4))
pred_mean.plot(ax=ax, label="Pronóstico SARIMA", color="#FF7043")
ax.fill_between(pred_IC.index, pred_IC.iloc[:, 0], pred_IC.iloc[:, 1], color="#FF7043", alpha=0.25)
plt.axvline(split_ts, color="gray", ls="--", lw=1)
plt.title("SARIMA — zoom a las últimas ~3 semanas (línea = corte train/test)")
plt.legend(); plt.show()

In [ ]:
def metricas(y_real, y_pred, etiqueta):
    y_real, y_pred = y_real.align(y_pred, join="inner")
    assert y_pred.index.equals(y_real.index) and len(y_pred) == len(test)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    mae  = mean_absolute_error(y_real, y_pred)
    lab  = y_real.index.hour.isin(list(HORAS_LAB))
    rmse_lab = np.sqrt(mean_squared_error(y_real[lab], y_pred[lab]))
    pos  = y_real > 0
    mape = (np.abs((y_real[pos] - y_pred[pos]) / y_real[pos])).mean() * 100
    print(f"[{etiqueta}]  RMSE={rmse:.2f}  MAE={mae:.2f}  "
          f"RMSE(06-20h)={rmse_lab:.2f}  MAPE(y>0)={mape:.1f}%   [horas-hombre/h]")
    return rmse

rmse_sarima = metricas(test, pred_mean, "SARIMA")

# Prophet

Prophet ajusta un modelo **aditivo** con estacionalidades múltiples. Aquí brilla: maneja de forma
**nativa** la estacionalidad **diaria** (doble pico) y la **semanal** (caída de fin de semana), sin
exógenas. `yearly_seasonality=False` (solo ~25 semanas de datos).

In [ ]:
train_Prophet = train.reset_index()
train_Prophet.columns = ["ds", "y"]
train_Prophet.tail()

## Modelo 1 — estacionalidad diaria + semanal

In [ ]:
model_prophet = Prophet(daily_seasonality=True, weekly_seasonality=True,
                        yearly_seasonality=False, interval_width=0.95)
model_prophet.fit(train_Prophet)

In [ ]:
future_data = model_prophet.make_future_dataframe(periods=PERIODOS_FUT, freq=FREQ)
pred_Prophet = model_prophet.predict(future_data)
pred_Prophet[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail()

In [ ]:
fig = model_prophet.plot_components(pred_Prophet, uncertainty=True, figsize=(9, 6))
_oscurecer(fig); plt.show()

In [ ]:
fig = model_prophet.plot(pred_Prophet, include_legend=True)
fig.set_size_inches(12, 4); plt.title("Prophet — Modelo 1 (pronóstico)")
_oscurecer(fig); plt.show()

In [ ]:
prophet_pred = pred_Prophet.set_index("ds")["yhat"].clip(lower=0)
y_pred_p1 = prophet_pred.reindex(test.index)
rmse_prophet1 = metricas(test, y_pred_p1, "Prophet M1")

## Modelo 2 — eventos de baja carga (festivos / cierres entre semana)

Donde el tutorial usaba "periodos de recesión", aquí modelamos los **días entre semana sin
actividad** (festivos MX / cierres) como *holidays* de Prophet. **No** incluimos fines de semana: ya
los captura `weekly_seasonality` (serían colineales).

In [ ]:
# Días laborales (Lun–Vie) con CERO horas-hombre = festivos/cierres (data-driven).
carga_dia = datos["HorasHombre"].groupby(datos.index.normalize()).sum()
es_laboral = carga_dia.index.dayofweek < 5
cierres = carga_dia.index[(carga_dia.values == 0) & es_laboral]
eventos = pd.DataFrame({"holiday": "cierre_entre_semana", "ds": cierres})
eventos["lower_window"] = 0
eventos["upper_window"] = 0
print(f"Eventos de baja carga (festivos/cierres entre semana): {len(eventos)}")
eventos

In [ ]:
model_prophet2 = Prophet(daily_seasonality=True, weekly_seasonality=True,
                         yearly_seasonality=False, interval_width=0.95,
                         holidays=eventos if len(eventos) else None)  # robusto si no hay cierres
model_prophet2.fit(train_Prophet)

In [ ]:
future_data2 = model_prophet2.make_future_dataframe(periods=PERIODOS_FUT, freq=FREQ)
pred_Prophet2 = model_prophet2.predict(future_data2)
pred_Prophet2.set_index("ds", inplace=True)

In [ ]:
fig = model_prophet2.plot(pred_Prophet2.reset_index(), include_legend=True)
fig.set_size_inches(12, 4); plt.title("Prophet — Modelo 2 (con eventos de baja carga)")
_oscurecer(fig); plt.show()

In [ ]:
int_conf_Prophet2 = pred_Prophet2[["yhat_lower", "yhat_upper"]].clip(lower=0)
int_conf_Prophet2.tail(PERIODOS_FUT).head()

In [ ]:
p2 = pred_Prophet2["yhat"].clip(lower=0).reindex(test.index)
ic2 = int_conf_Prophet2.reindex(test.index)

ax = datos["HorasHombre"].plot(label="Observaciones", figsize=(12, 5), alpha=0.6)
pred_mean.plot(ax=ax, label="SARIMA", color="#FF7043")
ax.fill_between(pred_IC.index, pred_IC.iloc[:, 0], pred_IC.iloc[:, 1], color="#FF7043", alpha=0.2)
p2.plot(ax=ax, label="Prophet M2", color="#4DD0E1")
ax.fill_between(test.index, ic2.iloc[:, 0], ic2.iloc[:, 1], color="#4DD0E1", alpha=0.2)
plt.title("SARIMA vs. Prophet — pronóstico de horas hombre (rango completo)")
plt.legend(); plt.show()

In [ ]:
ax = datos.loc[zoom:, "HorasHombre"].plot(label="Observaciones", figsize=(12, 5))
pred_mean.plot(ax=ax, label="SARIMA", color="#FF7043")
ax.fill_between(pred_IC.index, pred_IC.iloc[:, 0], pred_IC.iloc[:, 1], color="#FF7043", alpha=0.2)
p2.plot(ax=ax, label="Prophet M2", color="#4DD0E1")
ax.fill_between(test.index, ic2.iloc[:, 0], ic2.iloc[:, 1], color="#4DD0E1", alpha=0.2)
plt.axvline(split_ts, color="gray", ls="--", lw=1)
plt.title("SARIMA vs. Prophet — zoom a las últimas ~3 semanas")
plt.legend(); plt.show()

In [ ]:
rmse_prophet2 = metricas(test, p2, "Prophet M2")
print()
print(f"RMSE  SARIMA={rmse_sarima:.2f} | Prophet M1={rmse_prophet1:.2f} | "
      f"Prophet M2={rmse_prophet2:.2f}   [horas-hombre/h]")

## Conclusión y advertencias

- Se compara el **error puntual** (RMSE/MAE), no el ancho de banda: SARIMA usa IC analítico de
  espacio de estados y Prophet, muestreo — no son equivalentes.
- El **horizonte es de 168 h**: solo las primeras ~24–72 h tienen banda estrecha; los días 4–7 son
  indicativos. La banda inferior se recortó a 0 (conservadora).
- **Caveats:** la variable es un **proxy de presencia** (no horas de nómina); el *snapshot* es de
  ~6 meses (Nov 2025 – May 2026), por lo que solo son creíbles horizontes de **días a semanas** y no
  hay efecto anual estimable. La estacionalidad **semanal** la captura Prophet de forma nativa y
  SARIMA vía **exógenas de Fourier**.
- Para producción conviene **refrescar `ds_entradas`** y reentrenar; el orden SARIMA y `K_FOURIER`
  son ajustables según evolucione la operación.

# Apéndice — serie de horario laboral 06–20h (s=15) · solo comparación

⚠️ **Caveat de pseudo-continuidad:** al quitar las horas 21–05h, el *lag* estacional s=15 salta
huecos reales (noche y fin de semana), así que las distancias del modelo dejan de coincidir con el
tiempo de reloj. **No** es el pronóstico reportado; se incluye solo como comparación rápida.

In [ ]:
ent_lab = ent[ent["hora"].between(6, 20)]
hh_lab = ent_lab.groupby("ts")["usuario"].nunique()
rejilla_lab = pd.DatetimeIndex([
    d + pd.Timedelta(hours=h)
    for d in pd.date_range(hh_lab.index.min().floor("D"), hh_lab.index.max().floor("D"), freq="D")
    for h in HORAS_LAB
])
serie_lab = hh_lab.reindex(rejilla_lab).fillna(0).astype(float)
print(f"Serie laboral 06-20h: {len(serie_lab):,} puntos (s=15), {(serie_lab == 0).mean():.0%} ceros")

In [ ]:
dec_lab = seasonal_decompose(serie_lab.to_numpy(), model="additive", period=15, extrapolate_trend="freq")
fig = dec_lab.plot(); fig.set_size_inches(11, 6)
_oscurecer(fig); plt.suptitle("Descomposición — horario laboral 06-20h (s=15)"); plt.show()